In [1]:
import pandas as pd

# Load dataset
data = pd.read_csv("student_data.csv")
data.columns = ["StudentID", "Day", "Mood", "ShirtColor"]

# Compute initial mood distribution (first day only)
day1 = data[data["Day"] == 1]
initial_probs = day1["Mood"].value_counts(normalize=True).reindex(["H", "S"], fill_value=0)

# --- Count transitions between moods ---
transitions = {"H->H": 0, "H->S": 0, "S->H": 0, "S->S": 0}

for student, group in data.groupby("StudentID"):
    moods = group.sort_values("Day")["Mood"].tolist()
    for a, b in zip(moods[:-1], moods[1:]):
        transitions[f"{a}->{b}"] += 1

# --- Calculate transition probabilities ---
p_hh = transitions["H->H"] / max(1, (transitions["H->H"] + transitions["H->S"]))
p_hs = transitions["H->S"] / max(1, (transitions["H->H"] + transitions["H->S"]))
p_sh = transitions["S->H"] / max(1, (transitions["S->H"] + transitions["S->S"]))
p_ss = transitions["S->S"] / max(1, (transitions["S->H"] + transitions["S->S"]))

transition_df = pd.DataFrame(
    [[p_hh, p_hs],
     [p_sh, p_ss]],
    index=["H", "S"],
    columns=["H", "S"]
)

# --- Emission probabilities ---
emission_df = pd.crosstab(data["Mood"], data["ShirtColor"], normalize="index")
emission_df = emission_df.reindex(index=["H", "S"], columns=["R", "G", "B"], fill_value=0)

# Observation sequence
obs = ["R", "B", "G"]
moods = ["H", "S"]
sequence_results = []

# --- Compute sequence probabilities manually ---
for m1 in moods:
    for m2 in moods:
        for m3 in moods:
            prob = (
                initial_probs[m1] *
                emission_df.loc[m1, obs[0]] *
                transition_df.loc[m1, m2] *
                emission_df.loc[m2, obs[1]] *
                transition_df.loc[m2, m3] *
                emission_df.loc[m3, obs[2]]
            )
            sequence_results.append(((m1, m2, m3), prob))

# --- Sort and display ---
results = pd.DataFrame(sequence_results, columns=["Mood Sequence", "Probability"])
results = results.sort_values("Probability", ascending=False).reset_index(drop=True)

# Display neatly formatted results
print("\n=== Initial Probabilities ===")
for mood, val in initial_probs.items():
    print(f"P({mood}) = {val:.4f}")

print("\n=== Transition Matrix ===")
print(transition_df.round(4))

print("\n=== Emission Matrix ===")
print(emission_df.round(4))

print("\n=== Sequence Probabilities ===")
for _, row in results.iterrows():
    print(f"{row['Mood Sequence']}: {row['Probability']:.8f}")

best = results.iloc[0]
print("\nMost probable sequence:", best["Mood Sequence"])
print(f"Probability = {best['Probability']:.8f}")



=== Initial Probabilities ===
P(H) = 0.6000
P(S) = 0.4000

=== Transition Matrix ===
        H       S
H  0.6545  0.3455
S  0.4500  0.5500

=== Emission Matrix ===
ShirtColor       R       G       B
Mood                              
H           0.7193  0.2807  0.0000
S           0.0000  0.1395  0.8605

=== Sequence Probabilities ===
('H', 'S', 'H'): 0.01620474
('H', 'S', 'S'): 0.00984532
('H', 'H', 'S'): 0.00000000
('H', 'H', 'H'): 0.00000000
('S', 'H', 'H'): 0.00000000
('S', 'H', 'S'): 0.00000000
('S', 'S', 'H'): 0.00000000
('S', 'S', 'S'): 0.00000000

Most probable sequence: ('H', 'S', 'H')
Probability = 0.01620474
